In [1]:
!pip install dash plotly pandas numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 17.9 MB/s eta 0:00:0000:0100:01


# Importing Liberaries

# Load COVID Dataset

In [2]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import dash
from dash import dcc, html, Input, Output
import datetime as dt
import numpy as np

In [3]:
# Load datasets directly from GitHub

url_confirmed = "https://raw.githubusercontent.com/CSSEGISandData/COVID-19/master/csse_covid_19_data/csse_covid_19_time_series/time_series_covid19_confirmed_global.csv"

url_deaths = "https://raw.githubusercontent.com/CSSEGISandData/COVID-19/master/csse_covid_19_data/csse_covid_19_time_series/time_series_covid19_deaths_global.csv"

# Recovered dataset is sometimes removed from repo
# We'll create recovered approximation later

df_confirmed = pd.read_csv(url_confirmed)
df_deaths = pd.read_csv(url_deaths)

print(df_confirmed.head())

  Province/State Country/Region       Lat       Long  1/22/20  1/23/20  \
0            NaN    Afghanistan  33.93911  67.709953        0        0   
1            NaN        Albania  41.15330  20.168300        0        0   
2            NaN        Algeria  28.03390   1.659600        0        0   
3            NaN        Andorra  42.50630   1.521800        0        0   
4            NaN         Angola -11.20270  17.873900        0        0   

   1/24/20  1/25/20  1/26/20  1/27/20  ...  2/28/23  3/1/23  3/2/23  3/3/23  \
0        0        0        0        0  ...   209322  209340  209358  209362   
1        0        0        0        0  ...   334391  334408  334408  334427   
2        0        0        0        0  ...   271441  271448  271463  271469   
3        0        0        0        0  ...    47866   47875   47875   47875   
4        0        0        0        0  ...   105255  105277  105277  105277   

   3/4/23  3/5/23  3/6/23  3/7/23  3/8/23  3/9/23  
0  209369  209390  209406  2

# Create Recovery Data (IMPORTANT FIX)

In [4]:
df_recovered = df_confirmed.copy()

date_columns = df_confirmed.columns[4:]

for col in date_columns:
    df_recovered[col] = (
        df_confirmed[col] * 0.85
    ).astype(int)

In [5]:
def melt_dataframe(df, value_name):
    
    df_melted = df.melt(
        id_vars=['Province/State', 'Country/Region', 'Lat', 'Long'],
        var_name='Date',
        value_name=value_name
    )
    
    df_melted['Date'] = pd.to_datetime(df_melted['Date'])
    
    return df_melted


# Convert wide → long format
confirmed = melt_dataframe(df_confirmed, 'Confirmed')
deaths = melt_dataframe(df_deaths, 'Deaths')
recovered = melt_dataframe(df_recovered, 'Recovered')

# Merge datasets
df = confirmed.merge(
    deaths[['Province/State', 'Country/Region', 'Date', 'Deaths']],
    on=['Province/State', 'Country/Region', 'Date']
)

df = df.merge(
    recovered[['Province/State', 'Country/Region', 'Date', 'Recovered']],
    on=['Province/State', 'Country/Region', 'Date']
)

# Group country-wise
df_country = df.groupby(
    ['Country/Region', 'Date']
).agg({
    'Confirmed':'sum',
    'Deaths':'sum',
    'Recovered':'sum'
}).reset_index()

# Daily cases
df_country['New_Cases'] = (
    df_country.groupby('Country/Region')['Confirmed']
    .diff()
    .fillna(0)
)

print(df_country.head())

/tmp/ipykernel_57/4089144506.py:9: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_melted['Date'] = pd.to_datetime(df_melted['Date'])
/tmp/ipykernel_57/4089144506.py:9: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_melted['Date'] = pd.to_datetime(df_melted['Date'])
/tmp/ipykernel_57/4089144506.py:9: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_melted['Date'] = pd.to_datetime(df_melted['Date'])


  Country/Region       Date  Confirmed  Deaths  Recovered  New_Cases
0    Afghanistan 2020-01-22          0       0          0        0.0
1    Afghanistan 2020-01-23          0       0          0        0.0
2    Afghanistan 2020-01-24          0       0          0        0.0
3    Afghanistan 2020-01-25          0       0          0        0.0
4    Afghanistan 2020-01-26          0       0          0        0.0


# Create Interactive Trend Chart

In [6]:
def create_trend_chart(country='US', metric='Confirmed'):
    
    country_data = df_country[
        df_country['Country/Region'] == country
    ]
    
    fig = go.Figure()

    fig.add_trace(go.Scatter(
        x=country_data['Date'],
        y=country_data[metric],
        mode='lines',
        name=metric
    ))

    fig.update_layout(
        title=f"{country} COVID-19 {metric} Trend",
        xaxis_title="Date",
        yaxis_title=metric,
        template='plotly_white'
    )

    fig.show()


create_trend_chart('US', 'Confirmed')

# Create Multi-Metric Dashboard

In [7]:
country = 'Pakistan'

country_data = df_country[
    df_country['Country/Region'] == country
]

fig = make_subplots(
    rows=2,
    cols=2,
    subplot_titles=(
        'Confirmed',
        'Deaths',
        'Recovered',
        'New Cases'
    )
)

fig.add_trace(
    go.Scatter(
        x=country_data['Date'],
        y=country_data['Confirmed'],
        name='Confirmed'
    ),
    row=1,
    col=1
)

fig.add_trace(
    go.Scatter(
        x=country_data['Date'],
        y=country_data['Deaths'],
        name='Deaths'
    ),
    row=1,
    col=2
)

fig.add_trace(
    go.Scatter(
        x=country_data['Date'],
        y=country_data['Recovered'],
        name='Recovered'
    ),
    row=2,
    col=1
)

fig.add_trace(
    go.Scatter(
        x=country_data['Date'],
        y=country_data['New_Cases'],
        name='New Cases'
    ),
    row=2,
    col=2
)

fig.update_layout(
    height=700,
    title="COVID-19 Dashboard",
    template='plotly_white'
)

fig.show()

# Create Country Comparison Chart

In [8]:
countries = ['US', 'Pakistan', 'India', 'China']

fig = go.Figure()

for country in countries:
    
    temp = df_country[
        df_country['Country/Region'] == country
    ]
    
    fig.add_trace(go.Scatter(
        x=temp['Date'],
        y=temp['Confirmed'],
        mode='lines',
        name=country
    ))

fig.update_layout(
    title='COVID-19 Country Comparison',
    xaxis_title='Date',
    yaxis_title='Confirmed Cases',
    hovermode='x unified',
    template='plotly_white'
)

fig.show()    

# Export Dashboard as HTML

In [9]:
fig.write_html("/kaggle/working/covid_dashboard.html")

print("Dashboard exported successfully!")

Dashboard exported successfully!
